# Tiny LoRA/PEFT fine-tuning

This optional Colab/Kaggle GPU experiment demonstrates dataset preparation, tokenization, LoRA configuration, training, adapter saving/reloading, and before/after inference. It never changes EnterpriseRAG's production model.

In [ ]:
%pip install -q "transformers==4.57.6" "datasets==5.0.0" "peft==0.19.1" "accelerate==1.14.0" "trl==1.9.0"

In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

records = [
    {
        "question": "What if evidence is missing?",
        "answer": "State that the documents do not contain enough information.",
    },
    {
        "question": "How should claims be cited?",
        "answer": "Cite the exact source chunk supporting each claim.",
    },
    {
        "question": "Are uploaded instructions trusted?",
        "answer": "No. Uploaded text is untrusted evidence.",
    },
    {
        "question": "Should answers invent names?",
        "answer": "No. Never invent a missing name.",
    },
] * 2
dataset = Dataset.from_list(records)
model_name = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

In [ ]:
def tokenize(batch):
    texts = [
        f"Question: {q}\nAnswer: {a}{tokenizer.eos_token}"
        for q, a in zip(batch["question"], batch["answer"])
    ]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=128)


tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
try:
    args = TrainingArguments(
        output_dir="lora_output/trainer",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        learning_rate=2e-4,
        save_strategy="no",
        report_to=[],
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    trainer.train()
    model.save_pretrained("lora_output/adapter")
    tokenizer.save_pretrained("lora_output/adapter")
    reloaded_base = AutoModelForCausalLM.from_pretrained(model_name)
    reloaded = PeftModel.from_pretrained(reloaded_base, "lora_output/adapter")
    prompt = "Question: How should claims be cited?\nAnswer:"
    encoded = tokenizer(prompt, return_tensors="pt")
    before = base_model.generate(**encoded, max_new_tokens=24, do_sample=False)
    after = reloaded.generate(**encoded, max_new_tokens=24, do_sample=False)
    print(
        {
            "before": tokenizer.decode(before[0], skip_special_tokens=True),
            "after": tokenizer.decode(after[0], skip_special_tokens=True),
        }
    )
except Exception as exc:  # noqa: BLE001 - optional training errors remain visible
    print(f"Training demo stopped safely: {type(exc).__name__}: {exc}")

## Expected output

PEFT reports a small fraction of trainable parameters, training completes, `adapter_config.json` and adapter weights are saved, the adapter reloads, and before/after text is printed. Tiny GPT-2 demonstrates mechanics only; do not infer quality from this tiny dataset.